# Scratch: Test rapido de configuraciones VCP

Notebook liviano para probar parametros sin guardar en MLflow.
Solo muestra tablas de resultados.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import group_signals_into_patterns, simulate_trade

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

Imports OK


## Configuracion

Modifica los parametros aca y re-ejecuta las celdas de abajo.

In [ ]:
USE_VOLUME_CONTRACTION = True
VOLUME_RATIO_THRESHOLD = 1.5

SWING_CONFIG = ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False)

SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    "max_depth_pct": 0.35,
    "max_depth_atr": 7,        # None para desactivar
    "min_total_reduction": 0.80,
    "max_gap_between_contractions_days": None,
    "require_ascending_lows": True,
    "ascending_lows_tolerance": 0.1,
}

COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,
}

VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,
}

BREAKOUT_PARAMS = {
    "volume_method": "ratio",
    "volume_ratio_threshold": VOLUME_RATIO_THRESHOLD,
    "volume_lookback_days": 50,
    "require_volume_confirmation": True,
    "max_entry_distance_pct": 0.1,  # None para desactivar
}

RISK_PARAMS = {
    "max_stop_loss_pct": 0.05,
    "breakeven_r_multiple": 2.0,
    "trailing_sma_period": 20,
    "trailing_volume_factor": 1.5,
    "trailing_stop_method": "atr",
    "trailing_atr_period": 14,
    "trailing_atr_multiplier": 3.0,
    "max_bars_without_progress": 20,
    "min_progress_r": 0.5,
    "early_exit_days": 3,            # Minervini: salir si close < entry en los primeros N dias. None para desactivar
}

DATA_DIR = project_root / "data" / "csv"
TICKERS = sorted([p.stem for p in DATA_DIR.glob("*.csv")])
print(f"Tickers ({len(TICKERS)}): {TICKERS}")

## Ejecucion

In [59]:
def load_ohlc(ticker):
    return pd.read_csv(DATA_DIR / f"{ticker}.csv", parse_dates=["date"], index_col="date")


vol_params = VOLUME_CONTRACTION_PARAMS if USE_VOLUME_CONTRACTION else None
rows = []

for ticker in TICKERS:
    ohlc = load_ohlc(ticker)
    detector = ATRZigZagDetector(SWING_CONFIG)

    results = run_full_vcp_pipeline(
        ohlc=ohlc,
        swing_detector=detector,
        sequence_params=SEQUENCE_PARAMS,
        compression_params=COMPRESSION_PARAMS,
        breakout_params=BREAKOUT_PARAMS,
        volume_contraction_params=vol_params,
    )

    signals = {dt: sig for dt, sig in results.items() if sig is not None}
    patterns = group_signals_into_patterns(signals, risk_params=RISK_PARAMS)

    trades = []
    for p in patterns:
        trade = simulate_trade(ohlc, p, RISK_PARAMS)
        trade["pattern"] = p
        trades.append(trade)

    n_trades = len(trades)
    wins = sum(1 for t in trades if t["pnl_pct"] > 0)
    cum_ret = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1) if trades else 0.0
    avg_r = float(np.mean([t["r_multiple"] for t in trades])) if trades else 0.0

    rows.append({
        "ticker": ticker,
        "n_signals": len(signals),
        "n_patterns": len(patterns),
        "n_trades": n_trades,
        "wins": wins,
        "losses": n_trades - wins,
        "win_rate": wins / n_trades if n_trades > 0 else 0.0,
        "cum_return": cum_ret,
        "avg_r": avg_r,
    })
    status = f"{n_trades}T ({wins}W/{n_trades - wins}L) CR={cum_ret:+.1%}" if n_trades > 0 else "--"
    print(f"  {ticker}: {status}")

print("\nListo.")

  AAPL: 4T (4W/0L) CR=+64.7%
  AMZN: 10T (5W/5L) CR=+19.0%
  AVGO: 2T (0W/2L) CR=-12.5%
  BRK.B: 4T (0W/4L) CR=-13.4%
  COIN: 1T (0W/1L) CR=-8.5%
  GLD: 3T (1W/2L) CR=+12.8%
  GOOGL: 5T (4W/1L) CR=+18.9%
  HOOD: --
  IWM: 3T (1W/2L) CR=+15.0%
  JPM: 2T (1W/1L) CR=-0.9%
  META: 2T (1W/1L) CR=+18.7%
  MSFT: 7T (5W/2L) CR=+33.7%
  NVDA: 2T (0W/2L) CR=-16.4%
  PLTR: 3T (1W/2L) CR=+30.1%
  QLD: 3T (2W/1L) CR=-0.1%
  QQQ: 4T (4W/0L) CR=+17.4%
  SLV: 3T (0W/3L) CR=-13.4%
  SOFI: 2T (0W/2L) CR=-12.6%
  SPY: 4T (3W/1L) CR=+6.8%
  SQQQ: --
  TIL: --
  TLT: 3T (1W/2L) CR=-1.2%
  TQQQ: 1T (1W/0L) CR=+2.3%

Listo.


## Resultados

In [11]:
df = pd.DataFrame(rows)

# --- Totales ---
traded = df[df["n_trades"] > 0]
totals = pd.DataFrame([{
    "ticker": "TOTAL",
    "n_signals": int(df["n_signals"].sum()),
    "n_patterns": int(df["n_patterns"].sum()),
    "n_trades": int(df["n_trades"].sum()),
    "wins": int(df["wins"].sum()),
    "losses": int(df["losses"].sum()),
    "win_rate": float(traded["win_rate"].mean()) if len(traded) > 0 else 0.0,
    "cum_return": float(traded["cum_return"].mean()) if len(traded) > 0 else 0.0,
    "avg_r": float(traded["avg_r"].mean()) if len(traded) > 0 else 0.0,
}])
result = pd.concat([df, totals], ignore_index=True)

def color_returns(val):
    if isinstance(val, (int, float)):
        if val > 0: return "background-color: #27ae60; color: white"
        elif val < 0: return "background-color: #e74c3c; color: white"
    return ""

display(result.style.format({
    "win_rate": "{:.0%}",
    "cum_return": "{:+.1%}",
    "avg_r": "{:.2f}",
    "n_signals": "{:.0f}",
    "n_patterns": "{:.0f}",
    "n_trades": "{:.0f}",
    "wins": "{:.0f}",
    "losses": "{:.0f}",
}).map(color_returns, subset=["cum_return"]).background_gradient(
    subset=["win_rate"], cmap="RdYlGn", vmin=0, vmax=1
))

,ticker,n_signals,n_patterns,n_trades,wins,losses,win_rate,cum_return,avg_r
0,AAPL,6,5,5,4,1,80%,+61.8%,1.91
1,AMZN,11,8,8,3,5,38%,-0.8%,-0.08
2,AVGO,6,4,4,2,2,50%,+4.4%,0.20
3,BRK.B,0,0,0,0,0,0%,+0.0%,0.00
4,COIN,0,0,0,0,0,0%,+0.0%,0.00
5,GLD,5,3,3,1,2,33%,+11.1%,0.95
6,GOOGL,7,6,6,1,5,17%,-19.9%,-0.73
7,HOOD,0,0,0,0,0,0%,+0.0%,0.00
8,IWM,0,0,0,0,0,0%,+0.0%,0.00
9,JPM,5,4,4,3,1,75%,+28.6%,2.43
